# Build a battery RUL prediction dataset

This notebook creates a separate, labeled cycle-level dataset from `metadata.csv` and the prepared discharge time-series CSV files in `tmp/`. It numbers discharge cycles within each battery, identifies the first cycle with capacity at or below **1.4 Ah** as EOL, extracts voltage and temperature features, attaches the latest impedance `Re`/`Rct` measurements available by that discharge, and calculates `RUL = EOL_cycle - discharge_cycle`.

Run from the repository root. The output `rul_prediction_dataset.csv` contains only cycles up to and including an observed EOL; batteries that never reach the threshold in the available data are reported as right-censored in `rul_eol_summary.csv` and are not given invented RUL labels.

## 1. Imports and paths

The default paths are relative to the current working directory. Start Jupyter from the project root or change `BASE_DIR` to that folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
METADATA_PATH = BASE_DIR / "metadata.csv"
CYCLE_DATA_DIR = BASE_DIR / "tmp"
RUL_OUTPUT_PATH = BASE_DIR / "rul_prediction_dataset.csv"
EOL_SUMMARY_PATH = BASE_DIR / "rul_eol_summary.csv"
EOL_CAPACITY_AH = 1.4

if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {METADATA_PATH}. Start Jupyter from the project root "
        "or update BASE_DIR."
    )
if not CYCLE_DATA_DIR.is_dir():
    raise FileNotFoundError(f"Could not find the cycle CSV directory: {CYCLE_DATA_DIR}")

print(f"Metadata: {METADATA_PATH}")
print(f"Cycle CSV directory: {CYCLE_DATA_DIR}")
print(f"EOL capacity threshold: {EOL_CAPACITY_AH:.1f} Ah")

Metadata: c:\Users\prshi\OneDrive\Desktop\SOH-prediction-using-NASA-Dataset-master\SOH-prediction-using-NASA-Dataset-master\metadata.csv
Cycle CSV directory: c:\Users\prshi\OneDrive\Desktop\SOH-prediction-using-NASA-Dataset-master\SOH-prediction-using-NASA-Dataset-master\tmp
EOL capacity threshold: 1.4 Ah


## 2. Load and validate metadata

`test_id` identifies the original operation in the mixed charge/discharge/impedance sequence. It is used only to establish chronology and join measurements; the discharge-cycle number is calculated separately.

In [2]:
metadata = pd.read_csv(METADATA_PATH)
required_columns = {"type", "battery_id", "test_id", "filename", "Capacity", "Re", "Rct"}
missing_columns = required_columns.difference(metadata.columns)
if missing_columns:
    raise ValueError(f"metadata.csv is missing required columns: {sorted(missing_columns)}")

discharge_metadata = metadata.loc[
    metadata["type"].astype(str).str.lower().eq("discharge"),
    ["battery_id", "test_id", "filename", "Capacity"],
].copy()
discharge_metadata["test_id"] = pd.to_numeric(discharge_metadata["test_id"], errors="coerce")
discharge_metadata["Capacity"] = pd.to_numeric(discharge_metadata["Capacity"], errors="coerce")
if discharge_metadata[["battery_id", "test_id", "filename", "Capacity"]].isna().any().any():
    raise ValueError("Discharge metadata contains missing battery IDs, test IDs, filenames, or capacities.")

missing_files = [
    filename
    for filename in discharge_metadata["filename"].astype(str)
    if not (CYCLE_DATA_DIR / Path(filename).name).is_file()
]
if missing_files:
    raise FileNotFoundError(
        f"Missing {len(missing_files)} discharge CSV file(s); examples: {missing_files[:5]}"
    )

print(f"Discharge records: {len(discharge_metadata)}")
print(f"Batteries: {sorted(discharge_metadata['battery_id'].astype(str).unique())}")
display(discharge_metadata.head())

Discharge records: 636
Batteries: ['B0005', 'B0006', 'B0007', 'B0018']


,battery_id,test_id,filename,Capacity
1,B0005,1,00002.csv,1.856487
3,B0005,3,00004.csv,1.846327
5,B0005,5,00006.csv,1.835349
7,B0005,7,00008.csv,1.835263
9,B0005,9,00010.csv,1.834646


## 3. Number discharge cycles and identify EOL

Within each battery, records are sorted by `test_id`; `discharge_cycle` increments only over discharge records. EOL is the first discharge with measured capacity `<= 1.4 Ah`. A battery that never meets that threshold is right-censored: its EOL and RUL are unknown from the observed data.

In [3]:
discharge_metadata["battery_id"] = discharge_metadata["battery_id"].astype(str)
discharge_metadata = discharge_metadata.sort_values(
    ["battery_id", "test_id"]
).reset_index(drop=True)
discharge_metadata["discharge_cycle"] = (
    discharge_metadata.groupby("battery_id").cumcount() + 1
)

eol_summary_rows = []
for battery_id, battery_cycles in discharge_metadata.groupby("battery_id", sort=True):
    eol_records = battery_cycles.loc[battery_cycles["Capacity"] <= EOL_CAPACITY_AH]
    if eol_records.empty:
        eol_cycle = pd.NA
        eol_test_id = pd.NA
        eol_capacity_ah = np.nan
        status = "right_censored_no_observed_threshold_crossing"
    else:
        eol_record = eol_records.iloc[0]
        eol_cycle = int(eol_record["discharge_cycle"])
        eol_test_id = int(eol_record["test_id"])
        eol_capacity_ah = float(eol_record["Capacity"])
        status = "eol_observed"

    last_record = battery_cycles.iloc[-1]
    eol_summary_rows.append({
        "battery_id": battery_id,
        "eol_status": status,
        "eol_cycle": eol_cycle,
        "eol_test_id": eol_test_id,
        "eol_capacity_ah": eol_capacity_ah,
        "capacity_threshold_ah": EOL_CAPACITY_AH,
        "last_observed_discharge_cycle": int(last_record["discharge_cycle"]),
        "last_observed_capacity_ah": float(last_record["Capacity"]),
    })

eol_summary = pd.DataFrame(eol_summary_rows)
display(eol_summary)
display(discharge_metadata.groupby("battery_id")["discharge_cycle"].agg(["min", "max", "count"]))

,battery_id,eol_status,eol_cycle,eol_test_id,eol_capacity_ah,capacity_threshold_ah,last_observed_discharge_cycle,last_observed_capacity_ah
0,B0005,eol_observed,125,448,1.396701,1.4,168,1.325079
1,B0006,eol_observed,109,386,1.395164,1.4,168,1.185675
2,B0007,right_censored_no_observed_threshold_crossing,<NA>,<NA>,NaN,1.4,168,1.432455
3,B0018,eol_observed,97,236,1.396855,1.4,132,1.341051


,min,max,count
battery_id,,,
B0005,1,168,168
B0006,1,168,168
B0007,1,168,168
B0018,1,132,132


## 4. Extract discharge time-series features

`voltage_drop` is first measured voltage minus last measured voltage. `voltage_slope_v_per_s` is the least-squares voltage slope against elapsed time in seconds. `temperature_rise` is maximum measured temperature minus the first measured temperature. Features use samples from that discharge only.

In [4]:
def extract_discharge_features(csv_path):
    """Summarize voltage and temperature behavior for one discharge cycle."""
    measurements = pd.read_csv(csv_path)
    required_measurements = {"Time", "Voltage_measured", "Temperature_measured"}
    missing_measurements = required_measurements.difference(measurements.columns)
    if missing_measurements:
        raise ValueError(
            f"{csv_path} is missing required measurement columns: {sorted(missing_measurements)}"
        )

    measurements = measurements[
        ["Time", "Voltage_measured", "Temperature_measured"]
    ].apply(pd.to_numeric, errors="coerce").dropna()
    measurements = measurements.replace([np.inf, -np.inf], np.nan).dropna()
    if len(measurements) < 2:
        raise ValueError(f"{csv_path} has fewer than two valid time/voltage/temperature samples.")

    elapsed_seconds = measurements["Time"].to_numpy(dtype=float)
    voltage = measurements["Voltage_measured"].to_numpy(dtype=float)
    temperature = measurements["Temperature_measured"].to_numpy(dtype=float)
    if np.ptp(elapsed_seconds) <= 0:
        raise ValueError(f"{csv_path} does not contain a varying time measurement.")

    voltage_slope = np.polyfit(elapsed_seconds - elapsed_seconds[0], voltage, deg=1)[0]
    return {
        "voltage_drop": float(voltage[0] - voltage[-1]),
        "voltage_slope_v_per_s": float(voltage_slope),
        "temperature_rise": float(temperature.max() - temperature[0]),
    }


feature_rows = []
for row in discharge_metadata.itertuples(index=False):
    csv_path = CYCLE_DATA_DIR / Path(str(row.filename)).name
    features = extract_discharge_features(csv_path)
    feature_rows.append({
        "battery_id": row.battery_id,
        "test_id": row.test_id,
        "filename": row.filename,
        "discharge_cycle": row.discharge_cycle,
        "Capacity": row.Capacity,
        **features,
    })

discharge_features = pd.DataFrame(feature_rows)
print(f"Extracted features for {len(discharge_features)} discharge cycles.")
display(discharge_features.head())

Extracted features for 636 discharge cycles.


,battery_id,test_id,filename,discharge_cycle,Capacity,voltage_drop,voltage_slope_v_per_s,temperature_rise
0,B0005,1,00002.csv,1,1.856487,0.914322,-0.000206,14.652147
1,B0005,3,00004.csv,2,1.846327,0.889528,-0.000205,14.335646
2,B0005,5,00006.csv,3,1.835349,0.860736,-0.000200,14.084531
3,B0005,7,00008.csv,4,1.835263,0.874279,-0.000205,14.108068
4,B0005,9,00010.csv,5,1.834646,0.882802,-0.000208,14.140596


## 5. Attach only previously available impedance measurements

The metadata stores `Re` and `Rct` on impedance tests, not on discharge rows. To avoid future-data leakage, each discharge receives the most recent impedance measurements from the same battery whose `test_id` is at or before the discharge. Early cycles may have missing impedance values because no earlier impedance test was recorded.

In [5]:
impedance_metadata = metadata.loc[
    metadata["type"].astype(str).str.lower().eq("impedance"),
    ["battery_id", "test_id", "Re", "Rct"],
].copy()
impedance_metadata["battery_id"] = impedance_metadata["battery_id"].astype(str)
impedance_metadata["test_id"] = pd.to_numeric(impedance_metadata["test_id"], errors="coerce")
impedance_metadata["Re"] = pd.to_numeric(impedance_metadata["Re"], errors="coerce")
impedance_metadata["Rct"] = pd.to_numeric(impedance_metadata["Rct"], errors="coerce")
impedance_metadata = impedance_metadata.dropna(subset=["test_id", "Re", "Rct"])
impedance_metadata = impedance_metadata.rename(columns={"test_id": "impedance_test_id"})
if impedance_metadata.empty:
    raise ValueError("No impedance rows with valid test_id, Re, and Rct measurements were found.")

cycle_features = pd.merge_asof(
    discharge_features.sort_values("test_id"),
    impedance_metadata.sort_values("impedance_test_id"),
    left_on="test_id",
    right_on="impedance_test_id",
    by="battery_id",
    direction="backward",
    allow_exact_matches=True,
)

if len(cycle_features) != len(discharge_features):
    raise AssertionError("The impedance join changed the number of discharge-cycle rows.")
if (cycle_features["impedance_test_id"].notna() &
        (cycle_features["impedance_test_id"] > cycle_features["test_id"])).any():
    raise AssertionError("A future impedance measurement was matched to a discharge cycle.")

print(f"Discharge rows with a prior impedance measurement: {cycle_features['Re'].notna().sum()}")
print(f"Discharge rows without a prior impedance measurement: {cycle_features['Re'].isna().sum()}")
display(cycle_features[["battery_id", "discharge_cycle", "test_id", "impedance_test_id", "Re", "Rct"]].head(10))

Discharge rows with a prior impedance measurement: 579
Discharge rows without a prior impedance measurement: 57


,battery_id,discharge_cycle,test_id,impedance_test_id,Re,Rct
0,B0005,1,1,NaN,NaN,NaN
1,B0006,1,1,NaN,NaN,NaN
2,B0007,1,1,NaN,NaN,NaN
3,B0018,1,2,1.0,0.065158,0.095554
4,B0006,2,3,NaN,NaN,NaN
5,B0007,2,3,NaN,NaN,NaN
6,B0005,2,3,NaN,NaN,NaN
7,B0006,3,5,NaN,NaN,NaN
8,B0007,3,5,NaN,NaN,NaN
9,B0005,3,5,NaN,NaN,NaN


## 6. Create the labeled RUL dataset

For batteries with observed EOL, retain discharge cycles up to and including EOL and calculate `RUL = eol_cycle - discharge_cycle`. This keeps the target nonnegative and gives RUL 0 at EOL. Cycles after EOL and records from batteries without an observed threshold crossing are not included in the labeled training table.

In [6]:
observed_eol_cycles = {
    row.battery_id: int(row.eol_cycle)
    for row in eol_summary.itertuples(index=False)
    if pd.notna(row.eol_cycle)
}

labeled_cycles = cycle_features.copy()
labeled_cycles["eol_cycle"] = labeled_cycles["battery_id"].map(observed_eol_cycles)
labeled_cycles = labeled_cycles.loc[
    labeled_cycles["eol_cycle"].notna()
    & (labeled_cycles["discharge_cycle"] <= labeled_cycles["eol_cycle"])
].copy()
labeled_cycles["eol_cycle"] = labeled_cycles["eol_cycle"].astype(int)
labeled_cycles["RUL"] = labeled_cycles["eol_cycle"] - labeled_cycles["discharge_cycle"]

dataset_columns = [
    "battery_id",
    "discharge_cycle",
    "voltage_drop",
    "voltage_slope_v_per_s",
    "temperature_rise",
    "Re",
    "Rct",
    "RUL",
    "test_id",
]
rul_dataset = labeled_cycles[dataset_columns].sort_values(
    ["battery_id", "discharge_cycle"]
).reset_index(drop=True)

if rul_dataset.empty:
    raise ValueError("No labeled RUL rows were created. Check the capacity threshold and input data.")
if (rul_dataset["RUL"] < 0).any():
    raise AssertionError("RUL must not be negative in the labeled dataset.")
if rul_dataset.duplicated(["battery_id", "discharge_cycle"]).any():
    raise AssertionError("Duplicate battery/discharge-cycle rows were created.")

rul_dataset.to_csv(RUL_OUTPUT_PATH, index=False)
eol_summary.to_csv(EOL_SUMMARY_PATH, index=False)

print(f"Saved {len(rul_dataset)} labeled rows to {RUL_OUTPUT_PATH}")
print(f"Saved EOL status for all batteries to {EOL_SUMMARY_PATH}")
display(rul_dataset.head(10))
display(rul_dataset.groupby("battery_id").agg(
    labeled_cycles=("discharge_cycle", "count"),
    first_RUL=("RUL", "first"),
    last_RUL=("RUL", "last"),
))

Saved 331 labeled rows to c:\Users\prshi\OneDrive\Desktop\SOH-prediction-using-NASA-Dataset-master\SOH-prediction-using-NASA-Dataset-master\rul_prediction_dataset.csv
Saved EOL status for all batteries to c:\Users\prshi\OneDrive\Desktop\SOH-prediction-using-NASA-Dataset-master\SOH-prediction-using-NASA-Dataset-master\rul_eol_summary.csv


,battery_id,discharge_cycle,voltage_drop,voltage_slope_v_per_s,temperature_rise,Re,Rct,RUL,test_id
0,B0005,1,0.914322,-0.000206,14.652147,NaN,NaN,124,1
1,B0005,2,0.889528,-0.000205,14.335646,NaN,NaN,123,3
2,B0005,3,0.860736,-0.000200,14.084531,NaN,NaN,122,5
3,B0005,4,0.874279,-0.000205,14.108068,NaN,NaN,121,7
4,B0005,5,0.882802,-0.000208,14.140596,NaN,NaN,120,9
5,B0005,6,0.886487,-0.000209,14.229398,NaN,NaN,119,11
6,B0005,7,0.894651,-0.000211,14.235823,NaN,NaN,118,13
7,B0005,8,0.872697,-0.000205,13.803501,NaN,NaN,117,15
8,B0005,9,0.891618,-0.000210,13.891056,NaN,NaN,116,17
9,B0005,10,0.900819,-0.000213,13.954924,NaN,NaN,115,19


,labeled_cycles,first_RUL,last_RUL
battery_id,,,
B0005,125,124,0
B0006,109,108,0
B0018,97,96,0


## Dataset notes

- `battery_id`, `discharge_cycle`, and `test_id` are identifiers/grouping fields; do not pass them all blindly as model features. In particular, use battery-level splits when evaluating generalization to unseen batteries.
- `Capacity` and `eol_cycle` are intentionally excluded from the RUL feature table to avoid target leakage. They are used only to construct the labels and EOL summary.
- Missing `Re`/`Rct` values mean no earlier impedance measurement was available for that cycle. Keep these missing values explicit and handle them with a training-only imputer if a later model requires complete inputs.
- Batteries without an observed `<= 1.4 Ah` discharge are right-censored; their true EOL/RUL is unknown, so this notebook does not fabricate a label for them.